## Scenario 2: A cross-functional team with one data scientist working on an ML model


MLflow setup:
- tracking server: yes, local server
- backend store: sqlite database
- artifacts store: local filesystem

The experiments can be explored locally by accessing the local tracking server.

To run this example you need to launch the mlflow server locally by running the following command in your terminal:

`mlflow server --backend-store-uri sqlite:///backend.db`

In [1]:
import mlflow


mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://127.0.0.1:5000'


In [4]:
mlflow.search_experiments()

[<Experiment: artifact_location='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/3', creation_time=1717503591787, experiment_id='3', last_update_time=1717503591787, lifecycle_stage='active', name='my-cool-experiment-2', tags={}>,
 <Experiment: artifact_location='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/2', creation_time=1717495027972, experiment_id='2', last_update_time=1717495027972, lifecycle_stage='active', name='my-cool-experiment', tags={}>,
 <Experiment: artifact_location='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1717092736403, experiment_id='1', last_update_time=1717092736403, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/0', creation_time=1717092736400, experiment_id='0', last_update_time=1717092736400, lifecycle_stage='active', name='Default', tags={}>]

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2024/06/04 14:16:45 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.


default artifacts URI: 'mlflow-artifacts:/4/12389d728cbd499fa6f21a57e5949582/artifacts'


In [6]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1717507005268, experiment_id='4', last_update_time=1717507005268, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/3', creation_time=1717503591787, experiment_id='3', last_update_time=1717503591787, lifecycle_stage='active', name='my-cool-experiment-2', tags={}>,
 <Experiment: artifact_location='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/2', creation_time=1717495027972, experiment_id='2', last_update_time=1717495027972, lifecycle_stage='active', name='my-cool-experiment', tags={}>,
 <Experiment: artifact_location='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1717092736403, experiment_id='1', last_update_time=1717092736403, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='/Users/ruifspint

### Interacting with the model registry

In [7]:
from mlflow.tracking import MlflowClient


client = MlflowClient("http://127.0.0.1:5000")

In [12]:
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1717496891492, description='', last_updated_timestamp=1717503607927, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1717503603934, current_stage='None', description='', last_updated_timestamp=1717503603934, name='nyc-taxi-regressor', run_id='3fa4ba412d41431c910f495fc7021558', run_link='', source='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1/3fa4ba412d41431c910f495fc7021558/artifacts/models_mlflow', status='READY', status_message='', tags={}, user_id='', version='4'>,
  <ModelVersion: aliases=[], creation_timestamp=1717497008593, current_stage='Staging', description=('The model version <ModelVersion: aliases=[], '
  "creation_timestamp=1717497008593, current_stage='Staging', description=None, "
  "last_updated_timestamp=1717497939081, name='nyc-taxi-regressor', "
  "run_id='3fa4ba412d41431c910f495fc7021558', run_link=None, "
  "source='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-trac

In [21]:
client.search_experiments()[0]

<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1717507005268, experiment_id='4', last_update_time=1717507005268, lifecycle_stage='active', name='my-experiment-1', tags={}>

In [8]:
run_id = client.list_run_infos(experiment_id='4')[0].run_id

mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

AttributeError: 'MlflowClient' object has no attribute 'list_run_infos'